# 실습 3 — Image 분류 (9분 차시)

교재: [docs/03-image.md](../docs/03-image.md)

꽃 사진 데이터(tf_flowers)에서 **2종만 골라** MobileNetV2 **2단계 전이학습**을 돕니다.

> **실행 환경: Colab T4.** `런타임 → 런타임 유형 변경 → T4 GPU`로 먼저 바꾸고 시작하세요.
> GPU가 붙어 있지 않으면 아래 셀이 `assert`로 멈춥니다. 시험장과 같은 동작입니다.
> 강의 시간에 맞추려고 `EPOCHS` / `FINE_TUNE_EPOCHS`를 작게 잡아 두었습니다.

데이터가 2클래스인데도 `sigmoid`가 한 번도 나오지 않는 점을 눈여겨보세요.
`categorical`로 통일하면 클래스가 몇 개든 코드가 같습니다.
마지막 셀에서 **5클래스로 바꿔도 고칠 코드가 없다**는 것을 직접 확인합니다.

In [ ]:
import os
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print(f"TensorFlow {tf.__version__}")

## 0. 시험 환경 세팅

### GPU 연결 — 세션당 한 번

시험 환경의 GPU는 **자동으로 붙지 않습니다.** 연결 코드를 직접 실행해야 하고,
성능은 Colab 무료 T4와 비슷한 수준입니다.

세션이 유지되는 동안에는 **시작할 때 한 번이면 됩니다.**
단, **세션이 튕기면 다시 연결해야 합니다** — 재연결 시 CPU로 붙는 경우가 있습니다.

이걸 빼먹으면 코드는 **조용히 CPU로 떨어집니다.** 에러도 경고도 없습니다.
전이학습이 수십 배 느려지고, "왜 이렇게 오래 걸리지" 하다가 시험이 끝납니다.

그래서 `assert`로 멈춰 세웁니다. 아래 셀이 실패하면 런타임 유형부터 바꾸세요.
실전에서는 [templates/00_session_start.py](../templates/00_session_start.py)가 같은 일을 합니다.

In [ ]:
# Colab: 런타임 -> 런타임 유형 변경 -> T4 GPU
!nvidia-smi

gpus = tf.config.list_physical_devices('GPU')
print(f"GPU: {gpus}")
assert gpus, "GPU 미연결! 런타임 유형을 T4 GPU 로 바꾸고 다시 실행하세요"

### 제출 파일명

모델과 예측 결과를 `휴대폰번호_문제번호`로 저장해 제출합니다.
딥러닝 모델은 `.pkl`로 저장되지 않으므로, 문제가 형식을 지정하지 않았다면 `.keras`를 쓰되
**이름 규칙은 그대로 지킵니다.**

In [ ]:
PHONE = '01012345678'      # 세션당 한 번 — 본인 휴대폰 번호 (하이픈 없이)
PROB_NO = 1                # 문제마다 이 줄만

SUBMIT_CSV = f'{PHONE}_{PROB_NO}.csv'
SUBMIT_KERAS = f'{PHONE}_{PROB_NO}.keras'
print(f"제출 파일: {SUBMIT_CSV} / {SUBMIT_KERAS}")

## 1. 하이퍼파라미터

Phase 1과 Phase 2의 학습률 차이가 **100배**인 것이 이 차시의 핵심입니다.

In [ ]:
IMAGE_SIZE = (224, 224)         # MobileNetV2 기본 입력
INPUT_SHAPE = (224, 224, 3)
BATCH_SIZE = 32                 # OOM 나면 16 → 8

EPOCHS = 8                      # Phase 1 (강의용 축약. 실전은 30 + EarlyStopping)
LEARNING_RATE = 1e-3            # base 가 얼어 있으므로 크게 가도 된다

FINE_TUNE_EPOCHS = 5            # Phase 2
FINE_TUNE_LR = 1e-5             # 100배 낮춤 — 사전학습 가중치를 보존하기 위해
FINE_TUNE_LAYERS = 30           # 상위 30개 레이어만 해제

## 2. 데이터 로드

폴더 구조가 곧 라벨입니다. 하위 폴더명이 알파벳 순으로 클래스 0, 1, 2… 가 됩니다.

```
flower_photos/
  daisy/  dandelion/  roses/  sunflowers/  tulips/
```

강의 시간에 맞추려고 `class_names`로 **2종만** 골라 씁니다.
시험에서는 이 인자를 빼고 폴더에 있는 클래스를 전부 씁니다.

In [ ]:
url = 'https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz'
_p = pathlib.Path(tf.keras.utils.get_file('flower_photos', origin=url, untar=True))

# get_file 이 돌려주는 경로는 TF/Keras 버전마다 다릅니다(압축 파일 vs 해제된 폴더).
# 클래스 폴더를 직접 찾아 기준 경로를 확정합니다.
data_dir = _p if (_p / 'daisy').exists() else next(_p.rglob('flower_photos/daisy')).parent

print(f"경로: {data_dir}")
for cls in sorted(d.name for d in data_dir.iterdir() if d.is_dir()):
    print(f"  {cls}: {len(list((data_dir / cls).glob('*.jpg')))}장")

In [ ]:
# label_mode='categorical' 이 이진/다중 통일의 핵심.
# 기본값 'int' 나 'binary' 를 쓰면 클래스 수가 바뀔 때마다 손실 함수와 출력층을 고쳐야 한다.
# 강의용으로 2종만. None 으로 바꾸면 폴더에 있는 5클래스 전체가 됩니다.
CLASS_SUBSET = ['daisy', 'roses']

common = dict(label_mode='categorical', class_names=CLASS_SUBSET,
              image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
              seed=RANDOM_SEED, shuffle=True)

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset='training', **common)
val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset='validation', **common)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print(f"\n클래스 {NUM_CLASSES}개: {class_names}")

### 예측 순서 함정 — 지금 미리 대비해 둡니다

`shuffle=True`인 데이터셋은 **반복할 때마다 순서가 바뀝니다.**
그런데 `file_paths`는 원래 정렬 순서 그대로입니다.
나중에 `predict` 결과와 `file_paths`를 나란히 붙이면 **파일명과 예측이 어긋난 채 제출**됩니다.

검증 정확도는 정상으로 보이고 제출 점수만 랜덤 수준으로 나오기 때문에 발견이 매우 어렵습니다.
그래서 예측 전용 데이터셋을 **순서 보장되게** 따로 만듭니다.

In [ ]:
val_file_paths = list(val_ds.file_paths)
val_filenames = [os.path.basename(p) for p in val_file_paths]
val_true_names = [os.path.basename(os.path.dirname(p)) for p in val_file_paths]
val_true_idx = np.array([class_names.index(n) for n in val_true_names])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)     # GPU 계산 중 CPU 가 다음 배치 준비
val_ds = val_ds.prefetch(AUTOTUNE)


def _load_image(p):
    img = tf.io.read_file(p)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    return tf.image.resize(img, IMAGE_SIZE)


val_pred_ds = (tf.data.Dataset.from_tensor_slices(val_file_paths)
               .map(_load_image, num_parallel_calls=AUTOTUNE)
               .batch(BATCH_SIZE).prefetch(AUTOTUNE))

print(f"예측용 데이터셋 준비 완료 ({len(val_file_paths)}장, 순서 보장)")

In [ ]:
# 샘플 확인 — 데이터를 눈으로 보지 않고 학습을 시작하지 마세요
plt.figure(figsize=(14, 6))
for images, labels in train_ds.take(1):
    for i in range(min(10, len(images))):
        plt.subplot(2, 5, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[np.argmax(labels[i])], fontsize=9)
        plt.axis('off')
plt.suptitle('Sample Images'); plt.tight_layout(); plt.show()

## 3. 데이터 증강

증강 레이어를 **모델의 일부로** 넣으면 학습 때만 동작하고 추론 때는 자동으로 꺼집니다.

`RandomFlip`에 `"horizontal"`만 쓰는 이유가 있습니다.
상하 반전된 개 사진은 현실에 없습니다. 숫자·문자 이미지라면 좌우 반전조차 빼야 합니다
(6과 9가 뒤집히면 라벨이 틀려집니다).
**증강은 현실에서 일어날 수 있는 변형만** 넣는 것이 원칙입니다.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

# 증강 결과를 눈으로 확인
plt.figure(figsize=(12, 3))
for images, _ in train_ds.take(1):
    for i in range(6):
        augmented = data_augmentation(tf.expand_dims(images[0], 0), training=True)
        plt.subplot(1, 6, i + 1)
        plt.imshow(augmented[0].numpy().astype('uint8'))
        plt.axis('off')
plt.suptitle('같은 이미지의 증강 결과 6장'); plt.tight_layout(); plt.show()

## 4. 모델 구축

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=INPUT_SHAPE, weights='imagenet', include_top=False
)
base_model.trainable = False     # Phase 1: 동결

print(f"base model 레이어 {len(base_model.layers)}개")
print(f"base 파라미터 {base_model.count_params():,}개 (지금은 전부 동결)")

In [ ]:
inputs = tf.keras.Input(shape=INPUT_SHAPE)
x = data_augmentation(inputs)
x = tf.keras.layers.Rescaling(1. / 127.5, offset=-1)(x)   # MobileNetV2 는 -1~1 입력
x = base_model(x, training=False)                         # 동결 시 BatchNorm 통계 고정
x = tf.keras.layers.GlobalAveragePooling2D()(x)           # Flatten 대신 (파라미터 폭증 방지)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)   # 2든 7이든 동일

model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

trainable = np.sum([np.prod(v.shape) for v in model.trainable_weights])
print(f"학습 대상 파라미터: {trainable:,}개 - 전체의 극히 일부만 학습합니다")

네 줄만 짚고 갑니다.

- `Rescaling(1./127.5, offset=-1)` — MobileNetV2는 −1~1 입력으로 학습됐습니다. 0~1로 넣으면 성능이 떨어집니다.
- `base_model(x, training=False)` — 동결 상태에서 BatchNorm이 통계를 갱신하지 않게 합니다.
- `GlobalAveragePooling2D` — `Flatten`을 쓰면 파라미터가 폭증해 과적합합니다.
- `Dense(NUM_CLASSES, softmax)` — 이진이든 다중이든 동일. **`sigmoid`가 필요 없습니다.**

## 5. Phase 1 — Feature Extraction

base를 얼려 두고 새 분류 헤드만 학습합니다. 빠르게 수렴합니다.

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
]

# 긴 학습을 시작하기 직전입니다. 지금 Ctrl+S 로 저장하세요.
# 세션이 끊기는 사고는 대부분 이 구간에서 일어납니다.
history = model.fit(train_ds, validation_data=val_ds,
                    epochs=EPOCHS, callbacks=callbacks, verbose=1)

## 6. Phase 2 — Fine-Tuning

상위 30개 레이어만 풉니다.
하위 레이어는 선·모서리 같은 범용 특징이라 어떤 과제에나 유효하고,
상위로 갈수록 ImageNet에 특화된 특징이라 우리 데이터에 맞게 조정할 가치가 있습니다.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False

# trainable 을 바꿨으면 반드시 재컴파일. 안 하면 변경이 반영되지 않는다.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),   # 1e-3 → 1e-5
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

trainable = np.sum([np.prod(v.shape) for v in model.trainable_weights])
print(f"학습 대상 파라미터: {trainable:,}개 (Phase 1 대비 증가)")

history_ft = model.fit(train_ds, validation_data=val_ds,
                       epochs=FINE_TUNE_EPOCHS, callbacks=callbacks, verbose=1)

## 7. 학습 곡선

점선 이후가 Phase 2입니다. 여기서 한 단계 뛰어오르는 것이 정상입니다.

In [ ]:
acc = history.history['accuracy'] + history_ft.history['accuracy']
val_acc = history.history['val_accuracy'] + history_ft.history['val_accuracy']
loss = history.history['loss'] + history_ft.history['loss']
val_loss = history.history['val_loss'] + history_ft.history['val_loss']
split_at = len(history.history['accuracy']) - 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(acc, label='Train'); ax1.plot(val_acc, label='Val')
ax1.axvline(split_at, color='gray', ls='--', label='Fine-tune 시작')
ax1.set_title('Accuracy'); ax1.set_xlabel('epoch'); ax1.legend()
ax2.plot(loss, label='Train'); ax2.plot(val_loss, label='Val')
ax2.axvline(split_at, color='gray', ls='--')
ax2.set_title('Loss'); ax2.set_xlabel('epoch'); ax2.legend()
plt.tight_layout(); plt.show()

**곡선 읽는 법**

| 모양 | 원인 | 조치 |
|---|---|---|
| Train만 오르고 Val 정체 | 과적합 | 증강 강화, Dropout ↑, Phase 2 에폭 ↓ |
| 둘 다 낮음 | 과소적합 | 레이어 더 풀기, 더 큰 백본 |
| Phase 2에서 급락 | 학습률 과대 | `1e-5` 확인 |

## 8. 평가 — 순서 보장 데이터셋으로 예측

In [ ]:
probs = model.predict(val_pred_ds, verbose=1)
preds = np.argmax(probs, axis=1)
confs = probs.max(axis=1)

acc_final = accuracy_score(val_true_idx, preds)
f1_final = f1_score(val_true_idx, preds, average='macro')
print(f"Accuracy {acc_final:.4f} / F1(macro) {f1_final:.4f}")

grade = next((g for t, g in [(0.95, '매우 우수'), (0.90, '우수'),
                             (0.85, '양호'), (0.80, '보통')] if acc_final >= t), '미흡')
print(f"-> {grade}\n")
print(classification_report(val_true_idx, preds, target_names=class_names))

In [ ]:
cm = confusion_matrix(val_true_idx, preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix')
plt.tight_layout(); plt.show()

### 눈 검증 — 이미지를 직접 띄워 예측과 대조

초록 제목 = 실제와 일치, 빨강 = 불일치.
그림과 예측이 상식적으로 맞는지 확인하고 제출하세요.

In [ ]:
rng = np.random.RandomState(RANDOM_SEED)
pick = rng.choice(len(val_file_paths), min(12, len(val_file_paths)), replace=False)
plt.figure(figsize=(14, 8))
for j, k in enumerate(pick):
    plt.subplot(3, 4, j + 1)
    plt.imshow(tf.keras.utils.load_img(val_file_paths[k]))
    ok = preds[k] == val_true_idx[k]
    plt.title(f"P:{class_names[preds[k]]} / T:{val_true_names[k]}",
              fontsize=9, color='green' if ok else 'red')
    plt.axis('off')
plt.suptitle('Eyeball check: prediction vs actual')
plt.tight_layout(); plt.show()

### 틀린 것들을 보면 배울 게 있습니다

In [ ]:
wrong = np.where(preds != val_true_idx)[0]
print(f"오답 {len(wrong)}건 / 전체 {len(preds)}건")

if len(wrong):
    plt.figure(figsize=(14, 3))
    for i, idx in enumerate(wrong[:6]):
        img = tf.keras.utils.load_img(val_file_paths[idx], target_size=IMAGE_SIZE)
        plt.subplot(1, 6, i + 1)
        plt.imshow(img)
        plt.title(f"실제 {class_names[val_true_idx[idx]]}\n"
                  f"예측 {class_names[preds[idx]]} ({confs[idx]:.2f})", fontsize=8)
        plt.axis('off')
    plt.tight_layout(); plt.show()

## 9. 제출

In [ ]:
submission = pd.DataFrame({
    'filename': val_filenames,
    'predicted': [class_names[i] for i in preds],
    'confidence': confs,
})
submission.to_csv(SUBMIT_CSV, index=False)
model.save(SUBMIT_KERAS)

print(f"[저장] {SUBMIT_CSV} / {SUBMIT_KERAS}")
print('생성된 제출 파일:', [f for f in os.listdir('.') if f.startswith(PHONE)])
submission.head()

실제 시험에서 라벨 없는 `test/` 폴더가 주어지면 이렇게 처리합니다.

```python
exts = ('.jpg', '.jpeg', '.png')
filenames = sorted(f for f in os.listdir(TEST_DIR) if f.lower().endswith(exts))
images = np.array([
    tf.keras.utils.img_to_array(
        tf.keras.utils.load_img(os.path.join(TEST_DIR, f), target_size=IMAGE_SIZE))
    for f in filenames
])
preds = model.predict(images)
```

`sorted()`로 파일명을 정렬해 두면 채점 기준 파일과 순서를 맞추기 쉽습니다.

In [ ]:
# 위에서 이미 저장했습니다. 세션이 끊겨도 다시 학습하지 않도록,
# 오래 걸린 결과는 항상 파일로 떨궈 두세요.
print('완료. 마지막으로 Ctrl+S 를 눌러 노트북을 저장하세요.')

## 10. 클래스 수가 바뀌면 코드를 얼마나 고쳐야 하나

지금까지 2클래스로 돌렸습니다. 같은 폴더를 5클래스로 읽으면 무엇이 달라지는지 확인해 봅시다.

In [ ]:
ds5 = tf.keras.utils.image_dataset_from_directory(
    data_dir, label_mode='categorical', class_names=None,   # <- 이 인자만 뺐다
    validation_split=0.2, subset='training',
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, seed=RANDOM_SEED, shuffle=True)

print(f"2클래스: {class_names}")
print(f"5클래스: {ds5.class_names}")
print()
print(f"출력층    Dense({len(class_names)}, 'softmax')  ->  Dense({len(ds5.class_names)}, 'softmax')")
print("손실 함수  categorical_crossentropy  ->  categorical_crossentropy  (그대로)")
print("전처리     Rescaling / 증강           ->  그대로")
print("예측       np.argmax(probs, axis=1)   ->  그대로")

**고칠 코드가 없습니다.** `NUM_CLASSES`가 데이터에서 읽히고 출력층이 그 변수를 쓰기 때문입니다.

`sigmoid` + `binary_crossentropy`로 짰다면 여기서 출력층·손실 함수·예측 코드를 전부 고쳐야 했습니다.
시험장에서 데이터를 열어 보고 "아, 5개짜리네" 하는 순간에 무엇이 달라지는지가 이 차이입니다.

---
## 혼자 해 볼 것

1. Phase 2의 `FINE_TUNE_LR`을 `1e-3`으로 바꿔 돌려 보세요. 정확도가 어떻게 되나요?
2. `data_augmentation`을 통째로 빼면 train/val 곡선 간격이 어떻게 달라지나요?
3. `CLASS_SUBSET = None`으로 바꿔 5클래스 전체를 학습시켜 보세요.
   정확도와 학습 시간이 어떻게 달라지나요? (고쳐야 하는 코드는 그 한 줄뿐입니다)

---
마무리: [docs/04-exam-day.md](../docs/04-exam-day.md) — 시험 당일 체크리스트